# Diabetes Risk Screening Tool
**NextGen Institute · Machine Learning · Final Project** · Healthcare · Public Health · Classification

Team: _[names]_ · Dataset: [CDC Diabetes Health Indicators (UCI)](https://archive.ics.uci.edu/dataset/891)

## 1. Problem framing
- **Who has the problem?** Public-health teams and clinics: many people have diabetes or prediabetes and do not know it.
- **Decision the model supports:** *should this person see a doctor for a blood-sugar test?* A short questionnaire gives an early, cheap screen at population scale.
- **Target:** `Diabetes_binary` (0 = no diabetes, 1 = prediabetes or diabetes).
- **Metric that matters:** **recall** on the positive class. Missing a real case costs more than a false alarm, so accuracy is misleading here (the classes are imbalanced). We also report ROC-AUC, PR-AUC, precision and F1.
- **Important:** the tool is a *screening aid, not a diagnosis*.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
from sklearn.model_selection import train_test_split

from utils import *
from train import CANDIDATES, scores, tune, fit_with_threshold, RS

warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")
DATA_PATH = os.environ.get("DATA_PATH", "data/diabetes_binary_health_indicators_BRFSS2015.csv")
N_TUNE = int(os.environ.get("N_TUNE", 40000))   # rows used for the hyper-parameter search
N_ITER = int(os.environ.get("N_ITER", 8))
TARGET_RECALL = 0.80                            # screening goal

## 2. Data understanding
Source: CDC Behavioral Risk Factor Surveillance System (BRFSS) health survey, via UCI. Download the CSV (Kaggle copy `diabetes_binary_health_indicators_BRFSS2015.csv`) into `data/`, or let `ucimlrepo` fetch it.

| Column | Meaning |
|---|---|
| `Diabetes_binary` | **Target**: 0 no diabetes, 1 prediabetes/diabetes |
| `HighBP`, `HighChol`, `CholCheck` | high blood pressure, high cholesterol, cholesterol check in last 5 years |
| `BMI` | body mass index |
| `Smoker`, `HvyAlcoholConsump` | smoked 100+ cigarettes; heavy drinking |
| `Stroke`, `HeartDiseaseorAttack` | stroke; coronary heart disease or heart attack |
| `PhysActivity`, `Fruits`, `Veggies` | physical activity in past 30 days; daily fruit; daily vegetables |
| `AnyHealthcare`, `NoDocbcCost` | has health coverage; skipped doctor because of cost |
| `GenHlth` | general health, 1 (excellent) to 5 (poor) |
| `MentHlth`, `PhysHlth` | days of poor mental / physical health in last 30 days |
| `DiffWalk` | serious difficulty walking |
| `Sex`, `Age` | sex (0 female, 1 male); 13 age groups (1 = 18-24 ... 13 = 80+) |
| `Education`, `Income` | education level 1-6, income bracket 1-8 |

In [ ]:
df = load_data(DATA_PATH)
display(df.head())
print("shape:", df.shape, "| missing values:", int(df.isna().sum().sum()), "| duplicate rows:", int(df.duplicated().sum()))
display(df.dtypes.to_frame("dtype").T)
display(df.describe().T)
balance = df[TARGET].value_counts(normalize=True).rename("share")
display(balance)
balance.plot.bar(title="Class balance"); plt.show()

## 3. Exploratory data analysis

In [ ]:
# Univariate
df[CONTINUOUS + ORDINAL].hist(bins=25, figsize=(12, 6)); plt.tight_layout(); plt.show()

In [ ]:
# Bivariate: diabetes rate for each yes/no factor, and by age group / general health
rates = pd.Series({c: df.loc[df[c] == 1, TARGET].mean() for c in BINARY}).sort_values()
rates.plot.barh(figsize=(7, 5), title="Diabetes rate among people answering 'Yes' (or Male)")
plt.axvline(df[TARGET].mean(), color="red", ls="--", label="overall rate"); plt.legend(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
df.groupby("Age")[TARGET].mean().plot(marker="o", ax=ax[0], title="Diabetes rate by age group")
df.groupby("GenHlth")[TARGET].mean().plot(marker="o", ax=ax[1], title="... by general health (1=excellent)")
sns.boxplot(data=df, x=TARGET, y="BMI", ax=ax[2], showfliers=False); ax[2].set_title("BMI by diabetes status")
plt.tight_layout(); plt.show()

In [ ]:
# Multivariate: correlations
plt.figure(figsize=(11, 8)); sns.heatmap(df.corr(), cmap="coolwarm", center=0); plt.title("Correlation matrix"); plt.show()
df.corr()[TARGET].drop(TARGET).sort_values(ascending=False).head(8).to_frame("correlation with target")

### EDA insights (write 3-5 in your own words after looking at the charts)
### EDA insights
1. **Heart disease, stroke and walking difficulty carry the highest diabetes rates.** About 31-33% of people with any of these have diabetes or prediabetes, more than double the overall rate of about 14%. High blood pressure (~24%) and high cholesterol (~22%) are also well above average.
2. **Risk rises steeply with age and with poorer self-rated health.** The diabetes rate climbs from about 1% in the youngest age group to a peak of about 22% at ages 70-74, then dips slightly in the oldest groups. By general health it rises from about 3% ("excellent") to about 38% ("poor"), so one simple question carries a lot of signal.
3. **BMI is higher for people with diabetes.** The median BMI is about 31 for people with diabetes versus about 27 for people without, and the whole box sits higher. BMI is right-skewed with extreme outliers (up to 98), which is why the pipeline clips it to a plausible range.
4. **Healthy habits and risk go in the expected direction, with one surprise.** People who are physically active, eat fruit and vegetables have lower diabetes rates, but heavy drinkers have the lowest rate of all (about 6%). This is an association, not a cause, and probably reflects other factors (for example people with diabetes being advised to avoid alcohol). A model should not be read as advice here.
5. **No feature dominates, and correlated features are moderate.** The strongest correlations with the target are general health (0.29), high blood pressure (0.26), difficulty walking (0.22) and BMI (0.22). Some features move together (general health with physical health days and difficulty walking, income with education), but none are almost duplicates, so collinearity is not a serious problem for these models.

## 4. Preprocessing
- Remove exact duplicate rows **before** splitting, then split 80/20 with stratification, so the test set stays untouched.
- Everything else (imputation, BMI outlier clipping, scaling, the engineered `PoorHealthDays` = physical + mental bad days) lives in a scikit-learn `Pipeline` (`utils.make_pipeline`), so it is fitted on training data only: **no leakage**.

In [ ]:
df = df.drop_duplicates()
X, y = df[ALL_FEATURES], df[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)
print(len(X_train), "train rows /", len(X_test), "test rows | positive rate:", round(y_train.mean(), 3), round(y_test.mean(), 3))
make_pipeline(ALL_FEATURES, CANDIDATES["Logistic Regression"][0])

## 5-6. Modeling, tuning and cross-validation
A majority-class **baseline** plus three algorithms from the course: Logistic Regression, Random Forest and Gradient Boosting. Each is tuned with `RandomizedSearchCV` (5-fold stratified CV, ROC-AUC) on the **training data only**. Class imbalance is handled by tuning the decision threshold toward the recall goal (next section) instead of re-sampling, which keeps predicted probabilities meaningful.

In [ ]:
searches = {n: tune(n, X_train, y_train, N_TUNE, N_ITER) for n in CANDIDATES}
cv = pd.DataFrame({n: {"cv_roc_auc": s.best_score_, **{k.replace("model__", ""): v for k, v in s.best_params_.items()}}
                   for n, s in searches.items()}).T
cv

## 7. Evaluation on the untouched test set
For each model the threshold is chosen on a **validation split of the training data** (highest threshold that still reaches the recall goal), then applied once to the test set.

In [ ]:
baseline = make_pipeline(ALL_FEATURES, DummyClassifier(strategy="prior")).fit(X_train, y_train)
rows = {"Baseline (majority class)": scores(y_test, baseline.predict_proba(X_test)[:, 1], 0.5)}
final, thresholds = {}, {}
for n, s in searches.items():
    est = clone(CANDIDATES[n][0]).set_params(**{k.replace("model__", ""): v for k, v in s.best_params_.items()})
    _, thr, _, _ = fit_with_threshold(ALL_FEATURES, est, X_train, y_train, TARGET_RECALL)
    final[n] = make_pipeline(ALL_FEATURES, est).fit(X_train, y_train); thresholds[n] = thr
    rows[n] = scores(y_test, final[n].predict_proba(X_test)[:, 1], thr)
results = pd.DataFrame(rows).T
display(results.round(3))
best = max(searches, key=lambda n: searches[n].best_score_)
print("Best model by CV ROC-AUC:", best)

In [ ]:
proba = final[best].predict_proba(X_test)[:, 1]; thr = thresholds[best]
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ConfusionMatrixDisplay.from_predictions(y_test, proba >= thr, ax=ax[0], colorbar=False); ax[0].set_title(f"{best} @ threshold {thr:.2f}")
for n, m in final.items():
    RocCurveDisplay.from_predictions(y_test, m.predict_proba(X_test)[:, 1], name=n, ax=ax[1])
    PrecisionRecallDisplay.from_predictions(y_test, m.predict_proba(X_test)[:, 1], name=n, ax=ax[2])
fp, mp = calibration_curve(y_test, proba, n_bins=10)
ax[3].plot(mp, fp, marker="o", label=best); ax[3].plot([0, 1], [0, 1], "k--", label="perfect"); ax[3].set_title("Calibration"); ax[3].legend()
ax[3].set_xlabel("predicted risk"); ax[3].set_ylabel("observed rate"); plt.tight_layout(); plt.show()

In [ ]:
# Error analysis: who does the model miss (false negatives) vs. catch (true positives)?
pred = proba >= thr; pos = X_test[y_test == 1]
fn, tp = pos[~pred[y_test.values == 1]], pos[pred[y_test.values == 1]]
pd.DataFrame({"missed (FN) mean": fn.mean(), "caught (TP) mean": tp.mean()}).round(2).sort_values("caught (TP) mean", ascending=False).head(12)

### Reading the results
The majority-class baseline reaches 84.7% accuracy but has recall of 0, so it never finds a single person with diabetes. The best model, Gradient Boosting, reaches ROC-AUC 0.819 and, at a threshold of 0.14 chosen on validation data, catches 81.8% of positive cases in the untouched test set. Of 7,019 people with diabetes in the test set, it finds 5,741 and misses 1,278.

The cost is precision of 0.305. The model flags 18,820 people (about 41% of the test set), and 13,079 of them are false alarms, so roughly 7 in 10 flagged people do not have diabetes. That is acceptable for a screening tool, because a flag only means "get a blood test". Accuracy falls to 0.687 because the threshold is deliberately low. The three models are very close (ROC-AUC 0.811 to 0.819), so Gradient Boosting's advantage is small. The calibration curve follows the diagonal well up to a predicted risk of about 0.6; above that the model slightly under-estimates risk, so the risk score is a ranking tool more than an exact probability.

**Who does the model miss?** Compared with the cases it catches, the missed cases are people who look healthier on the survey. They have lower BMI (27.7 vs 33.1), fewer poor-health days (3.3 vs 9.2 physical days), better general health ratings (2.45 vs 3.51), far less high blood pressure (36% vs 84%) and high cholesterol (39% vs 74%), and slightly higher income. The model finds it hard to detect diabetes in people who have few of the usual risk factors.

## 8. Which questions matter? A short screening form (stretch goal)
Permutation importance ranks the questions. We then re-train using only the top *k* questions and see how much recall we keep.

In [ ]:
est = final[best].named_steps["model"]
m_val, _, X_v, y_v = fit_with_threshold(ALL_FEATURES, est, X_train, y_train, TARGET_RECALL)
Xs = X_v.sample(min(len(X_v), 20000), random_state=RS)
imp = permutation_importance(m_val, Xs, y_v.loc[Xs.index], scoring="roc_auc", n_repeats=3, random_state=RS, n_jobs=-1)
ranking = pd.Series(imp.importances_mean, index=ALL_FEATURES).sort_values(ascending=False)
ranking.head(12).iloc[::-1].plot.barh(title="Permutation importance (drop in ROC-AUC)"); plt.show()

k_rows = []
for k in [3, 5, 7, 10, 15, len(ALL_FEATURES)]:
    f = list(ranking.index[:k]); m, t, _, _ = fit_with_threshold(f, est, X_train, y_train, TARGET_RECALL)
    k_rows.append({"n_questions": k, **scores(y_test, m.predict_proba(X_test[f])[:, 1], t)})
k_curve = pd.DataFrame(k_rows).set_index("n_questions"); display(k_curve.round(3))
k_curve[["recall", "roc_auc"]].plot(marker="o", title="Recall / ROC-AUC vs. number of questions"); plt.show()

## 9. Save the pipelines for the web app
`train.py` repeats the steps above as a script and writes `models/model_full.joblib` and `models/model_short.joblib` (the short form used by `app.py`). Run it after the notebook:
```
python train.py --k 7
streamlit run app.py
```

## 10. Business insight and limitations
## 10. Business insight and limitations
**Recommendation.** A health organisation can use this questionnaire as a low-cost triage step: ask 7 short questions, then send the people flagged as high risk for a blood sugar (HbA1c) test. In our test set, flagging about 41% of people finds about 82% of cases, roughly twice as many cases as testing the same number of people at random. The next step is a pilot with follow-up blood tests to measure real-world performance.

**Limitations.**
- The data is a 2015 US self-reported survey, so the results may not transfer to other populations such as Egypt.
- Prediabetes and diabetes are grouped together as one class.
- About 7 in 10 flagged people are false alarms, and about 18% of real cases are missed, mostly people with few risk factors.
- The risk score is a model estimate and is slightly under-confident at high risk.
- It is a screening aid, not a diagnosis. Only a blood test and a doctor can confirm diabetes.

Data: CDC Diabetes Health Indicators, UCI Machine Learning Repository, https://archive.ics.uci.edu/dataset/891
